# Sprint 2 — 沉睡客回流行為分析

- 資料來源：`dormant_members.parquet`、`session01` / `session02`（各 6 個月）
- 觀察期：2023-09 ~ 2024-02
- 目標：從 session 行為資料中找出沉睡客的回流事件，計算 t0、彙總行為指標

> ⚠️ **Cell 4、7 產生輸出 Parquet，已輸出則不需重跑。Cell 1~3、5~6 為統計分析用。**  
> ⚠️ **原始 session CSV 已刪除，Cell 2、3、5、6 依賴原始 CSV，無法重跑。**

## Cell 1 — 引入資料

設定路徑（`dormant_members.parquet`、session01/02 CSV glob），建立輸出目錄，確認沉睡客人數。

> ℹ️ 不產生輸出檔案

In [9]:
import duckdb
import os

con = duckdb.connect()

DORMANT_PATH = 'output/sprint1/dormant_members.parquet'
S1_GLOB      = '91APP_Dataset(session01)/session01_*.csv'
S2_GLOB      = '91APP_Dataset(session02)/session02_*.csv'

os.makedirs('output/sprint2', exist_ok=True)

print('路徑設定完成')
print(f'  沉睡客  : {DORMANT_PATH}')
print(f'  Session01 : {S1_GLOB}')
print(f'  Session02 : {S2_GLOB}')

dormant_total = con.execute(f"SELECT COUNT(*) FROM read_parquet('{DORMANT_PATH}')").fetchone()[0]
print(f'\n沉睡客人數（從 dormant_members.parquet 讀取）：{dormant_total:,}')

路徑設定完成
  沉睡客  : output/sprint1/dormant_members.parquet
  Session01 : 91APP_Dataset(session01)/session01_*.csv
  Session02 : 91APP_Dataset(session02)/session02_*.csv

沉睡客人數（從 dormant_members.parquet 讀取）：1,112,087


## Cell 2 — Session01 & Session02 原始統計

計算 session01 / session02 原始總筆數，以及 `ShopMemberId` 為 NULL（匿名用戶）的比例。

> ℹ️ **純統計用，不產生輸出檔案，不需重跑。**  
> ⚠️ 依賴原始 session CSV（已刪除），若 CSV 不存在則無法執行。

In [10]:
for label, glob in [('Session01', S1_GLOB), ('Session02', S2_GLOB)]:
    raw = con.execute(f"""
    SELECT
        COUNT(*)                                           AS total_rows,
        COALESCE(SUM(CASE WHEN ShopMemberId IS NULL     THEN 1 END), 0) AS null_cnt,
        COALESCE(SUM(CASE WHEN ShopMemberId IS NOT NULL THEN 1 END), 0) AS non_null_cnt
    FROM read_csv_auto('{glob}')
    """).fetchone()

    total, null_cnt, non_null_cnt = raw
    print(f'=== {label} 原始統計 ===')
    print(f'  原始總筆數             : {total:>14,}')
    print(f'  ShopMemberId 為 NULL   : {null_cnt:>14,}  ({null_cnt/total*100:.2f}%)')
    print(f'  ShopMemberId 不為 NULL : {non_null_cnt:>14,}  ({non_null_cnt/total*100:.2f}%)')
    print()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== Session01 原始統計 ===
  原始總筆數             :     92,601,077
  ShopMemberId 為 NULL   :     29,962,672  (32.36%)
  ShopMemberId 不為 NULL :     62,638,405  (67.64%)



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== Session02 原始統計 ===
  原始總筆數             :     12,125,865
  ShopMemberId 為 NULL   :      2,424,644  (20.00%)
  ShopMemberId 不為 NULL :      9,701,221  (80.00%)



## Cell 3 — 篩選沉睡客行為資料（統計用）

將 session 與 `dormant_members` JOIN，統計篩選後的行為筆數與不重複沉睡客人數。

> ℹ️ **純統計用，不產生輸出檔案，不需重跑。**  
> ⚠️ 依賴原始 session CSV（已刪除），若 CSV 不存在則無法執行。  
> 💡 若只需要輸出結果，可跳過 Cell 2~3，直接執行 **Cell 4**。

In [11]:
for label, glob in [('Session01', S1_GLOB), ('Session02', S2_GLOB)]:
    result = con.execute(f"""
    WITH session AS (
        SELECT ShopMemberId, HitTime
        FROM read_csv_auto('{glob}')
        WHERE ShopMemberId IS NOT NULL
    ),
    dormant AS (
        SELECT ShopMemberId FROM read_parquet('{DORMANT_PATH}')
    )
    SELECT
        COUNT(*)                           AS total_rows,
        COALESCE(SUM(CASE WHEN ShopMemberId IS NULL     THEN 1 END), 0) AS null_cnt,
        COALESCE(SUM(CASE WHEN ShopMemberId IS NOT NULL THEN 1 END), 0) AS non_null_cnt,
        COUNT(DISTINCT ShopMemberId)       AS unique_members
    FROM session
    INNER JOIN dormant USING (ShopMemberId)
    """).fetchone()

    total, null_cnt, non_null_cnt, uniq = result
    print(f'=== {label} × 沉睡客篩選後 ===')
    print(f'  總行為筆數             : {total:>14,}')
    print(f'  ShopMemberId 為 NULL   : {null_cnt:>14,}')
    print(f'  ShopMemberId 不為 NULL : {non_null_cnt:>14,}')
    print(f'  不重複沉睡客人數       : {uniq:>14,}')
    print()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== Session01 × 沉睡客篩選後 ===
  總行為筆數             :      9,789,174
  ShopMemberId 為 NULL   :              0
  ShopMemberId 不為 NULL :      9,789,174
  不重複沉睡客人數       :        141,866



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== Session02 × 沉睡客篩選後 ===
  總行為筆數             :      1,409,749
  ShopMemberId 為 NULL   :              0
  ShopMemberId 不為 NULL :      1,409,749
  不重複沉睡客人數       :         36,481



## Cell 4 — 彙總沉睡客行為並輸出 Parquet

對 session01 / session02 各別：
- 以 `epoch_ms(HitTime)` 將毫秒時間戳轉為 TIMESTAMP
- 與 `dormant_members` JOIN，只保留沉睡客事件
- 彙總每人的首次回流時間（`t0`）、總事件數（`total_events`）、活躍天數（`active_days`）
- 輸出兩個 Parquet 備用

> 📄 **輸出**：`output/sprint2/session01_dormant_activity.parquet`（141,866 筆）  
> 📄 **輸出**：`output/sprint2/session02_dormant_activity.parquet`（36,481 筆）  
> ⚠️ 依賴原始 session CSV（已刪除），若 CSV 不存在則無法重跑。

In [12]:
outputs = {
    'Session01': (S1_GLOB, 'output/sprint2/session01_dormant_activity.parquet'),
    'Session02': (S2_GLOB, 'output/sprint2/session02_dormant_activity.parquet'),
}

for label, (glob, out_path) in outputs.items():
    con.execute(f"""
    COPY (
        WITH session AS (
            SELECT ShopMemberId, epoch_ms(HitTime) AS hit_ts
            FROM read_csv_auto('{glob}')
            WHERE ShopMemberId IS NOT NULL
        ),
        dormant AS (
            SELECT ShopMemberId FROM read_parquet('{DORMANT_PATH}')
        ),
        filtered AS (
            SELECT s.ShopMemberId, s.hit_ts
            FROM session s
            INNER JOIN dormant d USING (ShopMemberId)
        )
        SELECT
            ShopMemberId,
            MIN(hit_ts)                  AS t0,
            COUNT(*)                     AS total_events,
            COUNT(DISTINCT hit_ts::DATE) AS active_days
        FROM filtered
        GROUP BY ShopMemberId
        ORDER BY ShopMemberId
    ) TO '{out_path}' (FORMAT PARQUET)
    """)

    stats = con.execute(f"""
    SELECT
        COUNT(*)             AS members,
        MEDIAN(total_events) AS med_events,
        MIN(total_events)    AS min_events,
        MAX(total_events)    AS max_events,
        MEDIAN(active_days)  AS med_days,
        MIN(active_days)     AS min_days,
        MAX(active_days)     AS max_days
    FROM read_parquet('{out_path}')
    """).fetchone()

    print(f'=== {label} 沉睡客行為彙總 ===')
    print(f'  人數（回流沉睡客）       : {stats[0]:>10,}')
    print(f'  total_events 中位數      : {stats[1]:>10.1f}')
    print(f'  total_events 最小 / 最大 : {stats[2]:>6} / {stats[3]:>6}')
    print(f'  active_days  中位數      : {stats[4]:>10.1f}')
    print(f'  active_days  最小 / 最大 : {stats[5]:>6} / {stats[6]:>6}')
    print(f'  輸出：{out_path}')
    print()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== Session01 沉睡客行為彙總 ===
  人數（回流沉睡客）       :    141,866
  total_events 中位數      :       31.0
  total_events 最小 / 最大 :      1 /   7947
  active_days  中位數      :        3.0
  active_days  最小 / 最大 :      1 /    182
  輸出：output/sprint2/session01_dormant_activity.parquet



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== Session02 沉睡客行為彙總 ===
  人數（回流沉睡客）       :     36,481
  total_events 中位數      :       11.0
  total_events 最小 / 最大 :      1 /   3980
  active_days  中位數      :        2.0
  active_days  最小 / 最大 :      1 /    179
  輸出：output/sprint2/session02_dormant_activity.parquet



## Cell 5 — Session01 vs Session02 沉睡客重疊分析

確認 session01 與 session02 的 ShopId 組成，以及兩表 ShopMemberId 的跨表重疊情形。
結論：兩表屬於不同品牌（ShopId 完全不同），ShopMemberId 為 per-ShopId 雜湊，跨品牌 ID 不可比，重疊為 0。

> ℹ️ **純統計分析，不產生輸出檔案，不需重跑。**  
> ⚠️ 依賴原始 session CSV（已刪除），部分查詢無法重跑。

In [13]:
S1_OUT = 'output/sprint2/session01_dormant_activity.parquet'
S2_OUT = 'output/sprint2/session02_dormant_activity.parquet'

# 1. 確認各 Session 的 ShopId 與 Tunnel 分布
print('=== ShopId 確認（session01 vs session02）===')
for label, glob in [('Session01', S1_GLOB), ('Session02', S2_GLOB)]:
    shops = con.execute(f"""
        SELECT ShopId, COUNT(*) AS cnt
        FROM read_csv_auto('{glob}')
        GROUP BY ShopId ORDER BY cnt DESC
    """).df()
    print(f'\n{label} ShopId:')
    print(shops.to_string(index=False))

print('\n=== Tunnel（Web/App）分布 ===')
for label, glob in [('Session01', S1_GLOB), ('Session02', S2_GLOB)]:
    tunnels = con.execute(f"""
        SELECT Tunnel, COUNT(*) AS cnt,
               ROUND(COUNT(*)*100.0 / SUM(COUNT(*)) OVER (), 2) AS pct
        FROM read_csv_auto('{glob}')
        GROUP BY Tunnel ORDER BY cnt DESC
    """).df()
    print(f'\n{label}:')
    print(tunnels.to_string(index=False))

# 2. 重疊分析
overlap = con.execute(f"""
SELECT COUNT(*) FROM read_parquet('{S1_OUT}') AS s1
INNER JOIN read_parquet('{S2_OUT}') AS s2 USING (ShopMemberId)
""").fetchone()[0]

s1_total = con.execute(f"SELECT COUNT(*) FROM read_parquet('{S1_OUT}')").fetchone()[0]
s2_total = con.execute(f"SELECT COUNT(*) FROM read_parquet('{S2_OUT}')").fetchone()[0]

print(f'\n=== 跨 Session 重疊分析 ===')
print(f'  Session01 沉睡客人數 : {s1_total:>10,}')
print(f'  Session02 沉睡客人數 : {s2_total:>10,}')
print(f'  ShopMemberId 重疊人數 : {overlap:>10,}')
print()
if overlap == 0:
    print('  ⚠ 重疊為 0：session01 與 session02 屬於不同 ShopId（不同品牌），')
    print('    ShopMemberId 以 ShopId 為鹽值加密，同一真實用戶跨品牌的 ID 不同，')
    print('    因此兩表無法直接以 ShopMemberId JOIN 比較。')
    print('    → session01 和 session02 應視為兩個獨立的品牌用戶群體分析。')
else:
    pct_s1 = overlap / s1_total * 100
    pct_s2 = overlap / s2_total * 100
    print(f'  重疊佔 S01: {pct_s1:.2f}%  佔 S02: {pct_s2:.2f}%')

=== ShopId 確認（session01 vs session02）===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Session01 ShopId:
                  ShopId      cnt
RZSHERLBqjPGOUFO01RYew== 55425275
zXQPxhiL90nRa1XbvctRfA== 33060673
3WUOySTycJTK4Yeoza4Bjg==  4115129


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Session02 ShopId:
                  ShopId      cnt
hFwniXiB/Ev2ZPXeO630Sw== 12125865

=== Tunnel（Web/App）分布 ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Session01:
Tunnel      cnt   pct
   App 48424337 52.29
   Web 44176740 47.71


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Session02:
Tunnel     cnt   pct
   App 9257271 76.34
   Web 2868575 23.66
  None      19  0.00

=== 跨 Session 重疊分析 ===
  Session01 沉睡客人數 :    141,866
  Session02 沉睡客人數 :     36,481
  ShopMemberId 重疊人數 :          0

  ⚠ 重疊為 0：session01 與 session02 屬於不同 ShopId（不同品牌），
    ShopMemberId 以 ShopId 為鹽值加密，同一真實用戶跨品牌的 ID 不同，
    因此兩表無法直接以 ShopMemberId JOIN 比較。
    → session01 和 session02 應視為兩個獨立的品牌用戶群體分析。


## Cell 6 — Session01 內部 Web vs App 沉睡客分析

針對 session01 的沉睡客，分析 Web 與 App 管道的使用分布：
- 各管道的獨立用戶數與事件數
- 每人的管道使用分類（僅 Web / 僅 App / 雙管道）
- 雙管道用戶的首次使用管道順序（Web 先 vs App 先）

> ℹ️ **純統計分析，不產生輸出檔案，不需重跑。**  
> ⚠️ 依賴原始 session01 CSV（已刪除），無法重跑。

In [ ]:
S1_OUT = 'output/sprint2/session01_dormant_activity.parquet'

con.execute(f"""
CREATE OR REPLACE VIEW s1_dormant_tunnel AS
SELECT s.ShopMemberId, s.Tunnel, epoch_ms(s.HitTime) AS hit_ts
FROM read_csv_auto('{S1_GLOB}') AS s
INNER JOIN read_parquet('{S1_OUT}') AS d USING (ShopMemberId)
WHERE s.ShopMemberId IS NOT NULL
  AND s.Tunnel IN ('Web', 'App')
""")

# 1. Web vs App 基本統計
print('=== Session01 沉睡客 — Web vs App 基本統計 ===')
print(con.execute("""
    SELECT Tunnel,
           COUNT(DISTINCT ShopMemberId) AS unique_members,
           COUNT(*)                     AS total_events,
           COUNT(DISTINCT hit_ts::DATE) AS active_days_total
    FROM s1_dormant_tunnel
    GROUP BY Tunnel ORDER BY Tunnel
""").df().to_string(index=False))

# 2. 每人的管道使用分類
con.execute("""
CREATE OR REPLACE VIEW s1_member_tunnel_flag AS
SELECT ShopMemberId,
       MAX(CASE WHEN Tunnel='Web' THEN 1 ELSE 0 END) AS has_web,
       MAX(CASE WHEN Tunnel='App' THEN 1 ELSE 0 END) AS has_app
FROM s1_dormant_tunnel GROUP BY ShopMemberId
""")

web_only, app_only, both, total = con.execute("""
SELECT
    SUM(CASE WHEN has_web=1 AND has_app=0 THEN 1 END),
    SUM(CASE WHEN has_web=0 AND has_app=1 THEN 1 END),
    SUM(CASE WHEN has_web=1 AND has_app=1 THEN 1 END),
    COUNT(*)
FROM s1_member_tunnel_flag
""").fetchone()

print(f'\n=== 使用管道分類 ===')
print(f'  僅使用 Web        : {web_only:>10,}  ({web_only/total*100:.2f}%)')
print(f'  僅使用 App        : {app_only:>10,}  ({app_only/total*100:.2f}%)')
print(f'  Web + App 皆有    : {both:>10,}  ({both/total*100:.2f}%)')
print(f'  合計              : {total:>10,}')

# 3. Web+App 雙管道：誰先？
web_first, app_first, same_time = con.execute("""
WITH first_by_tunnel AS (
    SELECT ShopMemberId,
           MIN(CASE WHEN Tunnel='Web' THEN hit_ts END) AS first_web,
           MIN(CASE WHEN Tunnel='App' THEN hit_ts END) AS first_app
    FROM s1_dormant_tunnel
    GROUP BY ShopMemberId
    HAVING first_web IS NOT NULL AND first_app IS NOT NULL
)
SELECT
    COALESCE(SUM(CASE WHEN first_web < first_app THEN 1 END), 0),
    COALESCE(SUM(CASE WHEN first_app < first_web THEN 1 END), 0),
    COALESCE(SUM(CASE WHEN first_web = first_app THEN 1 END), 0)
FROM first_by_tunnel
""").fetchone()

print(f'\n=== Web+App 雙管道用戶首次管道（n={both:,}）===')
print(f'  先用 Web → 後轉 App : {web_first:>10,}  ({web_first/both*100:.2f}%)')
print(f'  先用 App → 後轉 Web : {app_first:>10,}  ({app_first/both*100:.2f}%)')
print(f'  同時首次出現         : {same_time:>10,}  ({same_time/both*100:.2f}%)')

## Cell 7 — 合併 Session01 + Session02 沉睡客行為資料

兩表無 ShopMemberId 重疊（不同品牌），直接 UNION ALL 合併為最終主表。
印出合併後統計：總人數、t0 範圍、events / active_days 分布。

> 📄 **輸出**：`output/sprint2/dormant_activity_merged.parquet`（178,347 筆）

In [ ]:
S1_OUT    = 'output/sprint2/session01_dormant_activity.parquet'
S2_OUT    = 'output/sprint2/session02_dormant_activity.parquet'
MERGED_OUT = 'output/sprint2/dormant_activity_merged.parquet'

# 兩表無重疊（不同 ShopId），直接 UNION ALL
con.execute(f"""
COPY (
    SELECT ShopMemberId, t0, total_events, active_days FROM read_parquet('{S1_OUT}')
    UNION ALL
    SELECT ShopMemberId, t0, total_events, active_days FROM read_parquet('{S2_OUT}')
    ORDER BY ShopMemberId
) TO '{MERGED_OUT}' (FORMAT PARQUET)
""")

stats = con.execute(f"""
SELECT
    COUNT(*)               AS total_members,
    MIN(t0)                AS earliest_t0,
    MAX(t0)                AS latest_t0,
    MEDIAN(total_events)   AS med_events,
    MIN(total_events)      AS min_events,
    MAX(total_events)      AS max_events,
    MEDIAN(active_days)    AS med_days,
    MIN(active_days)       AS min_days,
    MAX(active_days)       AS max_days
FROM read_parquet('{MERGED_OUT}')
""").fetchone()

print(f'=== 合併後統計 ===')
print(f'  總人數（S01+S02）       : {stats[0]:>10,}')
print(f'  最早 t0                 : {stats[1]}')
print(f'  最晚 t0                 : {stats[2]}')
print(f'  total_events 中位數     : {stats[3]:>10.1f}')
print(f'  total_events 最小 / 最大: {stats[4]:>6} / {stats[5]:>6}')
print(f'  active_days  中位數     : {stats[6]:>10.1f}')
print(f'  active_days  最小 / 最大: {stats[7]:>6} / {stats[8]:>6}')
print(f'\n  輸出：{MERGED_OUT}')